In [ ]:
from pathlib import Path
from typing import Optional
from collections import Counter
import json
import re

import pandas as pd
import numpy as np


def find_repo_root(start: Optional[Path] = None, repo_name: str = "masters_thesis_sdg") -> Path:
    """
    Walk upward from the current working directory until the repository root is found.
    """
    if start is None:
        start = Path.cwd().resolve()

    current = start
    while True:
        if current.name == repo_name:
            return current
        if current.parent == current:
            raise FileNotFoundError(
                f"Could not find repo root '{repo_name}' starting from {start}"
            )
        current = current.parent


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


REPO_ROOT = find_repo_root()

idx_file_path = (
    REPO_ROOT
    / "data"
    / "processed"
    / "lai2023"
    / "onuw_transcripts_ready"
    / "index_cleaned.json"
)

RESULTS_DIR = REPO_ROOT / "results" / "voting"

index_data = load_json(idx_file_path)

print("REPO_ROOT:", REPO_ROOT)
print("Index games:", len(index_data))
print("RESULTS_DIR:", RESULTS_DIR)

REPO_ROOT: C:\Users\annab\Documents\GitHub\masters_thesis_sdg
Index games: 191
RESULTS_DIR: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\results\voting


In [ ]:
def build_ground_truth_lookup(index_data):
    """
    Builds two lookup tables:
    1. source-aware: (source, session_name, game_key)
    2. fallback:     (session_name, game_key)

    The fallback is useful if older result JSONs do not contain source.
    """
    lookup_with_source = {}
    lookup_without_source = {}

    for game in index_data:
        source = game.get("source")
        session_name = game.get("session_name")
        game_key = game.get("game_key")

        if session_name is None or game_key is None:
            continue

        lookup_without_source[(session_name, game_key)] = game

        if source is not None:
            lookup_with_source[(source, session_name, game_key)] = game

    return lookup_with_source, lookup_without_source


ground_truth_lookup_source, ground_truth_lookup_fallback = build_ground_truth_lookup(index_data)

print("GT lookup with source:", len(ground_truth_lookup_source))
print("GT lookup fallback:", len(ground_truth_lookup_fallback))

GT lookup with source: 191
GT lookup fallback: 191


In [ ]:
def strip_prompt_prefix(prompt_family):
    """
    Accepts both 'v3' and 'prompt_v3'.
    Returns 'v3'.
    """
    prompt_family = str(prompt_family)
    return prompt_family[len("prompt_"):] if prompt_family.startswith("prompt_") else prompt_family


def discover_prompt_run_dirs(results_dir, llm, prompt_family):
    """
    Finds all prompt run directories for a given prompt family.

    Example:
        prompt_family = "v3"

    Matches:
        prompt_v3/
        prompt_v3_run_1/
        prompt_v3_run_2/
        prompt_v3_run_3/
        ...

    Does not require the number of runs to be fixed.
    """
    model_dir = results_dir / llm

    if not model_dir.exists():
        # useful when llm is a partial name, e.g. "unsloth_gemma-4-31B-it"
        matches = [d for d in results_dir.iterdir() if d.is_dir() and llm in d.name]

        if len(matches) == 1:
            model_dir = matches[0]
            print(f"Resolved LLM directory to: {model_dir.name}")
        elif len(matches) > 1:
            raise ValueError(
                "Multiple matching LLM directories found:\n"
                + "\n".join(str(m) for m in matches)
            )
        else:
            raise FileNotFoundError(f"Could not find model directory for llm={llm}")

    prompt_family = strip_prompt_prefix(prompt_family)

    pattern = re.compile(
        rf"^prompt_{re.escape(prompt_family)}(?:_run_(\d+))?$"
    )

    run_dirs = []

    for d in model_dir.iterdir():
        if not d.is_dir():
            continue

        match = pattern.match(d.name)
        if match:
            run_num = match.group(1)
            run_num = int(run_num) if run_num is not None else 0

            run_dirs.append({
                "prompt_dir": d,
                "prompt_version": d.name,
                "run": run_num,
            })

    run_dirs = sorted(run_dirs, key=lambda x: x["run"])

    if not run_dirs:
        raise FileNotFoundError(
            f"No prompt directories found for prompt family '{prompt_family}' under {model_dir}"
        )

    return run_dirs

###### MODEL NAME AND PROMPT CONFIGURATION ######
llm = "unsloth_gemma-4-31B-it"
prompt_family = "v4"

prompt_run_dirs = discover_prompt_run_dirs(
    RESULTS_DIR,
    llm=llm,
    prompt_family=prompt_family,
)

for item in prompt_run_dirs:
    print(item["run"], item["prompt_version"], item["prompt_dir"])

Resolved LLM directory to: unsloth_gemma-4-31B-it-unsloth-bnb-4bit
1 prompt_v3_run_1 C:\Users\annab\Documents\GitHub\masters_thesis_sdg\results\voting\unsloth_gemma-4-31B-it-unsloth-bnb-4bit\prompt_v3_run_1


In [ ]:
NO_WEREWOLF_LABELS = {
    "no werewolf",
}


def normalize_vote(vote):
    if vote is None:
        return None
    return str(vote).strip()


def is_no_werewolf_vote(vote):
    vote = normalize_vote(vote)
    if vote is None:
        return False
    return vote.lower() in NO_WEREWOLF_LABELS


def canonicalize_player_vote(chosen_vote, player_names):
    """
    Exact match first, then case-insensitive match.
    Returns canonical player name or None.
    """
    chosen_vote = normalize_vote(chosen_vote)

    if chosen_vote is None:
        return None

    if chosen_vote in player_names:
        return chosen_vote

    norm_map = {
        str(p).strip().lower(): p
        for p in player_names
    }

    return norm_map.get(chosen_vote.lower(), None)


def get_ground_truth_game(res_data, lookup_with_source, lookup_fallback):
    source = res_data.get("source")
    session_name = res_data.get("session_name")
    game_key = res_data.get("game_key")

    if source is not None:
        gt_game = lookup_with_source.get((source, session_name, game_key))
        if gt_game is not None:
            return gt_game

    return lookup_fallback.get((session_name, game_key))


def evaluate_llm_prompt_dir(prompt_dir, lookup_with_source, lookup_fallback):
    """
    Evaluates all JSON result files under one prompt directory.
    Jointly includes Youtube and Ego4D because we use rglob("*.json").
    """
    result_files = sorted(prompt_dir.rglob("*.json"))

    total_games_evaluated = 0
    correct_werewolf_votes = 0
    correct_circle_votes = 0
    attempted_circle_votes = 0
    invalid_votes = 0
    failed_parses = 0
    missing_ground_truth = 0

    per_file_rows = []

    for res_file in result_files:
        res_data = load_json(res_file)

        source = res_data.get("source")
        session_name = res_data.get("session_name")
        game_key = res_data.get("game_key")

        gt_game = get_ground_truth_game(
            res_data,
            lookup_with_source,
            lookup_fallback,
        )

        if gt_game is None:
            missing_ground_truth += 1
            per_file_rows.append({
                "path": str(res_file),
                "source": source,
                "session_name": session_name,
                "game_key": game_key,
                "status": "missing_ground_truth",
                "chosen_vote": None,
                "is_correct": np.nan,
            })
            continue

        total_games_evaluated += 1

        validation_info = res_data.get("validation", {}) or {}
        parsed_output = res_data.get("parsed_output", {}) or {}

        is_valid = validation_info.get("is_valid", False)
        chosen_vote_raw = (
            validation_info.get("chosen_vote")
            or parsed_output.get("chosen_vote")
        )

        if not is_valid or not chosen_vote_raw:
            failed_parses += 1
            per_file_rows.append({
                "path": str(res_file),
                "source": source,
                "session_name": session_name,
                "game_key": game_key,
                "status": "failed_parse",
                "chosen_vote": chosen_vote_raw,
                "is_correct": np.nan,
            })
            continue

        player_names = gt_game["player_names"]
        end_roles = gt_game["end_roles"]

        chosen_player = canonicalize_player_vote(chosen_vote_raw, player_names)

        is_correct = False
        status = None

        # Case 1: LLM voted for a player
        if chosen_player is not None:
            player_idx = player_names.index(chosen_player)
            voted_player_end_role = end_roles[player_idx]

            if voted_player_end_role == "Werewolf":
                correct_werewolf_votes += 1
                is_correct = True

            status = "player_vote"

        # Case 2: LLM voted no werewolf / circle
        elif is_no_werewolf_vote(chosen_vote_raw):
            attempted_circle_votes += 1

            if "Werewolf" not in end_roles:
                correct_circle_votes += 1
                is_correct = True

            status = "circle_vote"

        # Case 3: invalid vote
        else:
            invalid_votes += 1
            status = "invalid_vote"

        per_file_rows.append({
            "path": str(res_file),
            "source": source,
            "session_name": session_name,
            "game_key": game_key,
            "status": status,
            "chosen_vote": chosen_vote_raw,
            "canonical_player_vote": chosen_player,
            "end_roles": end_roles,
            "is_correct": is_correct if status != "invalid_vote" else np.nan,
        })

    valid_game_count = total_games_evaluated - failed_parses - invalid_votes
    total_correct = correct_werewolf_votes + correct_circle_votes

    accuracy = (
        total_correct / valid_game_count
        if valid_game_count > 0
        else np.nan
    )

    summary = {
        "prompt_version": prompt_dir.name,
        "total_files_found": len(result_files),
        "total_games_evaluated": total_games_evaluated,
        "missing_ground_truth": missing_ground_truth,
        "correct_werewolf_votes": correct_werewolf_votes,
        "correct_circle_votes": correct_circle_votes,
        "total_attempted_circles": attempted_circle_votes,
        "failed_parses": failed_parses,
        "invalid_votes": invalid_votes,
        "valid_votes_cast": valid_game_count,
        "total_llm_wins": total_correct,
        "accuracy": accuracy,
    }

    per_file_df = pd.DataFrame(per_file_rows)

    return summary, per_file_df

In [ ]:
run_summaries = []
all_file_rows = []

for item in prompt_run_dirs:
    prompt_dir = item["prompt_dir"]

    summary, per_file_df = evaluate_llm_prompt_dir(
        prompt_dir,
        ground_truth_lookup_source,
        ground_truth_lookup_fallback,
    )

    summary["run"] = item["run"]
    summary["prompt_dir"] = str(prompt_dir)

    per_file_df["run"] = item["run"]
    per_file_df["prompt_version"] = item["prompt_version"]

    run_summaries.append(summary)
    all_file_rows.append(per_file_df)

llm_run_summary_df = pd.DataFrame(run_summaries).sort_values("run")
llm_file_level_df = pd.concat(all_file_rows, ignore_index=True)

display(llm_run_summary_df)

,prompt_version,total_files_found,total_games_evaluated,missing_ground_truth,correct_werewolf_votes,correct_circle_votes,total_attempted_circles,failed_parses,invalid_votes,valid_votes_cast,total_llm_wins,accuracy,accuracy_percent,run,prompt_dir
0,prompt_v3_run_1,191,191,0,81,0,0,0,0,191,81,0.424084,42.408377,1,C:\Users\annab\Documents\GitHub\masters_thesis...


# Human baseline

In [ ]:
def is_missing_human_vote(raw_votes):
    if raw_votes is None:
        return True

    if isinstance(raw_votes, str):
        return raw_votes.strip().upper() in {"NA", "N/A", ""}

    if isinstance(raw_votes, list):
        if len(raw_votes) == 0:
            return True

        normalized = [str(v).strip().upper() for v in raw_votes]
        return any(v in {"NA", "N/A", ""} for v in normalized)

    return False


total_games_in_index = len(index_data)
valid_human_games = 0
human_correct_werewolf_kills = 0
human_correct_circle_votes = 0
human_attempted_circle_votes = 0

for game in index_data:
    raw_votes = game.get("voting_outcome", [])

    if is_missing_human_vote(raw_votes):
        continue

    votes = [int(v) for v in raw_votes]
    end_roles = game.get("end_roles", [])

    valid_human_games += 1

    vote_counts = Counter(votes)

    if not vote_counts:
        continue

    max_votes = max(vote_counts.values())

    # ONUW rule: a player must receive at least 2 votes to be eliminated.
    if max_votes >= 2:
        eliminated_players = [
            p_idx
            for p_idx, count in vote_counts.items()
            if count == max_votes
        ]

        wolf_killed = any(
            end_roles[p_idx] == "Werewolf"
            for p_idx in eliminated_players
        )

        if wolf_killed:
            human_correct_werewolf_kills += 1

    else:
        human_attempted_circle_votes += 1

        if "Werewolf" not in end_roles:
            human_correct_circle_votes += 1


total_human_wins = human_correct_werewolf_kills + human_correct_circle_votes
human_win_rate = (
    total_human_wins / valid_human_games
    if valid_human_games > 0
    else np.nan
)

human_summary_df = pd.DataFrame([{
    "total_games_in_index": total_games_in_index,
    "valid_human_games": valid_human_games,
    "correct_werewolf_kills": human_correct_werewolf_kills,
    "correct_circle_votes": human_correct_circle_votes,
    "total_attempted_circles": human_attempted_circle_votes,
    "total_human_wins": total_human_wins,
    "human_win_rate": human_win_rate,
    "human_win_rate_percent": human_win_rate * 100,
}])

display(human_summary_df)

,total_games_in_index,valid_human_games,correct_werewolf_kills,correct_circle_votes,total_attempted_circles,total_human_wins,human_win_rate,human_win_rate_percent
0,191,171,77,7,12,84,0.491228,49.122807


In [ ]:
comparison_with_human_df = llm_run_summary_df[
    [
        "prompt_version",
        "run",
        "valid_votes_cast",
        "total_llm_wins",
        "accuracy",
        "accuracy_percent",
        "failed_parses",
        "invalid_votes",
        "total_attempted_circles",
    ]
].copy()

comparison_with_human_df["human_win_rate"] = human_win_rate
comparison_with_human_df["human_win_rate_percent"] = human_win_rate * 100
comparison_with_human_df["delta_vs_human_percent"] = (
    comparison_with_human_df["accuracy_percent"]
    - comparison_with_human_df["human_win_rate_percent"]
)

display(comparison_with_human_df)

,prompt_version,run,valid_votes_cast,total_llm_wins,accuracy,accuracy_percent,failed_parses,invalid_votes,total_attempted_circles,human_win_rate,human_win_rate_percent,delta_vs_human_percent
0,prompt_v3_run_1,1,191,81,0.424084,42.408377,0,0,0,0.491228,49.122807,-6.71443


In [ ]:
#save
OUT_DIR = REPO_ROOT / "results" / "analysis" / "llm_vote_accuracy"
OUT_DIR.mkdir(parents=True, exist_ok=True)

llm_run_summary_df.to_csv(
    OUT_DIR / f"{llm}_{prompt_family}_run_summary.csv",
    index=False,
)

llm_file_level_df.to_csv(
    OUT_DIR / f"{llm}_{prompt_family}_file_level.csv",
    index=False,
)

across_runs_summary.to_csv(
    OUT_DIR / f"{llm}_{prompt_family}_across_runs_summary.csv",
    index=False,
)

human_summary_df.to_csv(
    OUT_DIR / "human_baseline_summary.csv",
    index=False,
)

comparison_with_human_df.to_csv(
    OUT_DIR / f"{llm}_{prompt_family}_vs_human.csv",
    index=False,
)

print("Saved to:", OUT_DIR)